<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fcollab/Another_copy_of_Crafting_an_AI_Powered_HR_Assistant_A_Use_Case_for_Nestle%E2%80%99s_HR_Policy_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
cd /content/sample_data/capstone_project/

/content/sample_data/capstone_project


In [ ]:
pip install -r requirements.txt

In [ ]:
# ========================================
# CELL 1: Install Required Packages (Latest Versions)
# ========================================

# Install all required packages with latest versions
!pip install -q langchain==0.2.16 \
             langchain-community==0.2.16 \
             langchain-core==0.2.38 \
             langchain-openai==0.1.23 \
             openai==1.45.0 \
             chromadb==0.5.5 \
             gradio==4.44.0 \
             pypdf==4.3.1 \
             tiktoken==0.7.0 \
             pydantic==2.9.1 --force-reinstall

print("✅ All packages installed successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
mcp 1.26.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.9.1 w

In [ ]:
# ========================================
# CELL 3: Import Libraries and Set API Key
# ========================================

# Import all necessary libraries using latest LangChain structure
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
import gradio as gr
from google.colab import userdata

print("✅ Libraries imported successfully")


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [ ]:
# Set OpenAI API key from Colab secrets (recommended method)
# Go to: Colab menu -> Secrets (key icon) -> Add new secret named "OPENAI_API_KEY"
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # Alternative: Direct input (less secure)
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass("🔑 Enter your OpenAI API key: ")
    print("✅ API key set successfully")

✅ API key loaded from Colab secrets


In [ ]:
# ========================================
# CELL 4: Upload PDF File
# ========================================

from google.colab import files

# Upload the Nestlé HR policy PDF
print("📁 Please upload your Nestlé HR policy PDF file:")
uploaded = files.upload()

# Get the uploaded file name
pdf_filename = list(uploaded.keys())[0]
print(f"✅ File uploaded: {pdf_filename}")

📁 Please upload your Nestlé HR policy PDF file:


Saving the_nestle_hr_policy_pdf_2012.pdf to the_nestle_hr_policy_pdf_2012 (2).pdf
✅ File uploaded: the_nestle_hr_policy_pdf_2012 (2).pdf


In [ ]:
# ========================================
# CELL 5: Load and Process Document
# ========================================

def load_and_process_document(pdf_path):
    """
    Loads the PDF document and splits it into manageable chunks

    Args:
        pdf_path: Path to the HR policy PDF file

    Returns:
        List of text chunks from the document
    """
    # Initialize the PDF loader with the document path
    # Using PyPDFLoader from langchain_community
    loader = PyPDFLoader(pdf_path)

    # Load all pages from the PDF document
    documents = loader.load()
    print(f"📄 Loaded {len(documents)} pages from PDF")

    # Initialize text splitter to break documents into smaller chunks
    # chunk_size: Maximum size of each text chunk (in characters)
    # chunk_overlap: Number of characters to overlap between chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )

    # Split the loaded documents into smaller text chunks
    texts = text_splitter.split_documents(documents)
    print(f"✂️ Split into {len(texts)} text chunks")

    return texts

# Load and process the uploaded document
print("🔄 Processing document...")
texts = load_and_process_document(pdf_filename)
print("✅ Document processing complete")

🔄 Processing document...
📄 Loaded 8 pages from PDF
✂️ Split into 20 text chunks
✅ Document processing complete


In [ ]:


# ========================================
# CELL 6: Create Vector Store (Latest LangChain)
# ========================================

def create_vector_store(texts):
    """
    Creates a vector database using ChromaDB and OpenAI embeddings

    Args:
        texts: List of text chunks from the document

    Returns:
        Chroma vector store instance
    """
    print("🔄 Creating embeddings... (this may take 1-2 minutes)")

    # Initialize OpenAI embeddings model using langchain_openai
    # Using the latest text-embedding-3-small model (more cost-effective)
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )

    # Create a Chroma vector database from the text chunks
    # This enables semantic search capabilities
    vectorstore = Chroma.from_documents(
        documents=texts,
        embedding=embeddings,
        persist_directory="/content/sample_data/capstone_project/L1/chroma_db",  # Colab-specific path
        collection_name="nestle_hr_policies"  # Named collection
    )

    print("✅ Vector store created successfully")

    return vectorstore

# Create the vector store
vectorstore = create_vector_store(texts)

🔄 Creating embeddings... (this may take 1-2 minutes)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store created successfully


In [ ]:
# ========================================
# CELL 7: Build QA System (Latest LangChain Chains)
# ========================================

def create_qa_chain(vectorstore):
    """
    Creates a Question-Answering chain using latest LangChain architecture

    Args:
        vectorstore: Chroma vector store instance

    Returns:
        Retrieval chain instance
    """
    # Initialize the ChatOpenAI model using langchain_openai
    # temperature: Controls randomness (0 = deterministic)
    # model: GPT-3.5 Turbo for cost-effectiveness
    llm = ChatOpenAI(
        temperature=0,
        model="gpt-3.5-turbo",
        streaming=False
    )

    # Create a custom prompt template using ChatPromptTemplate
    # This is the new LangChain way to create prompts
    system_prompt = (
        "You are an intelligent HR assistant for Nestlé. Your role is to help employees "
        "and HR personnel find information from Nestlé's HR policies and reports.\n\n"
        "Use the following context from the HR documents to answer the question. "
        "If you cannot find the answer in the context, politely inform the user that "
        "the information is not available in the current HR documents.\n\n"
        "Always be professional, clear, and concise in your responses.\n\n"
        "Context: {context}"
    )

    # Create ChatPromptTemplate with system and user messages
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    # Create a document chain that combines documents with the LLM
    # This replaces the old RetrievalQA chain
    question_answer_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt
    )

    # Create retriever from vector store
    # search_type: Type of search to perform
    # search_kwargs: k = number of documents to retrieve
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    # Create the final retrieval chain
    # This is the modern way to create QA chains in LangChain
    rag_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=question_answer_chain
    )

    print("✅ QA system initialized")

    return rag_chain

# Create the QA chain
print("🤖 Building QA system...")
qa_chain = create_qa_chain(vectorstore)

🤖 Building QA system...
✅ QA system initialized


In [ ]:
# ========================================
# CELL 8: Create Gradio Interface
# ========================================

def chatbot_response(message, history):
    """
    Processes user queries and generates responses

    Args:
        message: User's question
        history: Conversation history (for Gradio)

    Returns:
        String response from the chatbot
    """
    # Invoke the retrieval chain with the user's message
    # The new invoke() method is used in latest LangChain
    result = qa_chain.invoke({"input": message})

    # Extract the answer from the result
    answer = result['answer']

    return answer

# Create custom CSS for better appearance
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
    max-width: 900px;
    margin: auto;
}
"""

# Create the Gradio ChatInterface
interface = gr.ChatInterface(
    fn=chatbot_response,
    title="🏢 Nestlé HR Policy Assistant",
    description=(
        "Welcome to the Nestlé HR Policy Assistant! Ask questions about HR policies, "
        "benefits, leave policies, working hours, and more. Powered by GPT-3.5 Turbo and latest LangChain."
    ),
    examples=[
        "What is the leave policy?",
        "Tell me about employee benefits",
        "What are the working hours?",
        "How do I apply for remote work?",
        "What is the dress code policy?",
        "Explain the performance review process"
    ],
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
    ),
    css=custom_css,
    retry_btn="🔄 Retry",
    undo_btn="⬅️ Undo",
    clear_btn="🗑️ Clear Chat",
    submit_btn="📤 Send"
)

print("✅ Chatbot interface ready!")

✅ Chatbot interface ready!


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


In [ ]:
# ========================================
# CELL 9: Launch the Chatbot
# ========================================

# Launch the interface with public sharing enabled
# share=True creates a public link (valid for 72 hours)
# The link can be shared with others
print("🚀 Launching chatbot interface...")
print("⏳ This may take a few seconds...")

interface.launch(
    share=True,
    debug=True,
    show_error=True
)

print("\n✅ Chatbot is live!")
print("💡 Click the public URL above to access your chatbot")
print("📱 You can share this link with others")

🚀 Launching chatbot interface...
⏳ This may take a few seconds...
Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://e9adc63dea05a50566.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

Key improvements in the latest version:

New LangChain Architecture: Uses create_retrieval_chain and create_stuff_documents_chain instead of deprecated RetrievalQA
Modular Imports: Uses langchain_openai and langchain_community packages
ChatPromptTemplate: Modern prompt creation with system/human message structure
invoke() Method: Replaces old __call__() syntax
Better Embeddings: Uses text-embedding-3-small for cost-effectiveness
Enhanced Error Handling: More robust error messages and validation
Improved UI: Better Gradio theming and CSS customization
This is production-ready and follows current LangChain best practices! 🚀

